## 1️⃣ Importation & Exploration


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv("bank-transaction.csv")
df=pd.read_csv("bank-transaction.csv")
df.columns=df.columns.str.lower().str.strip()
df.head()
df.dtypes
df.shape
df.info()
df.describe()


In [ ]:
v_manque=df.isnull().sum()/df.shape[0]*100
v_manque
plt.bar(v_manque.index,v_manque.values)
plt.xticks(rotation=90)
for ind,vl in enumerate(v_manque.values):
    plt.text(ind,vl,str(vl.round(2)),ha="center",va="bottom")
plt.xlabel("colonne")
plt.ylabel("valeurs_manque")
plt.title("valeurs_manque par colonne")
plt.show()

In [ ]:
duplicate=df[df.duplicated(subset=["transaction_id"],keep=False)]
duplicate=duplicate.sort_values(by="transaction_id")
duplicate=duplicate.groupby("transaction_id")["date_transaction"].nunique()
duplicate

## 2️⃣ Nettoyage des données



In [ ]:
df.drop_duplicates(subset="transaction_id",keep="first",inplace=True)
df["date_transaction"] = pd.to_datetime(df["date_transaction"],errors="coerce",dayfirst=True)
df["montant"]=df["montant"].astype(str)
df["montant"]=df["montant"].str.replace(",",".")
df["montant"]=pd.to_numeric(df["montant"],errors="coerce").astype(float)
df["solde_avant"]=df["solde_avant"].astype(str)
df[df["solde_avant"].str.contains("EUR")]
df["solde_avant"]=df["solde_avant"].str.replace(" EUR","")
df["solde_avant"]=pd.to_numeric(df["solde_avant"]).astype(float)
df["devise"]=df["devise"].str.upper().str.strip()
df["segment_client"]=df["segment_client"].str.capitalize()
df["agence"]=df["agence"].str.strip()
df.info()

In [ ]:
mode_date=df["date_transaction"].mode()[0]
mode_agenge=df["agence"].mode()[0]
mode_segment=df["segment_client"].mode()[0]
df["score_credit_client"]=df["score_credit_client"].fillna(df["score_credit_client"].median())
df["agence"]=df["agence"].fillna(mode_agenge)
df["segment_client"]=df["segment_client"].fillna(mode_segment)
df["date_transaction"]=df["date_transaction"].fillna(mode_date)
df=df.drop(columns="taux_interet",errors="ignore")

## 3️⃣ Détection & Traitement des Valeurs Aberrantes



In [ ]:
Q1=df["montant"].quantile(0.25)
Q3=df["montant"].quantile(0.75)
IQR=Q3-Q1
val_max=Q3+1.5*IQR
val_min=Q1-1.5*IQR
anomalie_montant=(df["montant"]<val_min)|(df["montant"]>val_max)


In [ ]:
Q1_1=df["score_credit_client"].quantile(0.25)
Q3_3=df["score_credit_client"].quantile(0.75)
IQR_score=Q3_3-Q1_1
v_max=Q3_3+1.5*IQR_score
v_min=Q1_1-1.5*IQR_score
anomalie_score_credit = (df["score_credit_client"] < v_min) | (df["score_credit_client"] > v_max) 
anomalie_score=(df["score_credit_client"]< 0) | (df["score_credit_client"]>850)
df["is_anomalie"]= anomalie_montant | anomalie_score_credit

 ## 4️⃣ Feature Engineering

In [ ]:
df["annee"]=df["date_transaction"].dt.year
df["mois"]=df["date_transaction"].dt.month
df["trimestre"]=df["date_transaction"].dt.quarter
df["semaine-jour"]=df["date_transaction"].dt.dayofweek

In [ ]:
df["montant_eur_verifie"]=(df["montant"]/df["taux_change_eur"])
comparer=(df["montant_eur"]).compare(df["montant_eur_verifie"])
comparer

In [ ]:
def risque(ca):
    if ca >=700: 
        return "Low"
    elif ca>=580:
        return "Meduim"
    else:
        return "High"
df["categorie_risque"]=df["score_credit_client"].apply(risque)
df["type_operation"]

In [ ]:
total_credit=df[df["type_operation"]=="Credit"].groupby("client_id")["montant"].sum()
total_debit=df[df["type_operation"]=="Debit"].groupby("client_id")["montant"].sum()
solde=pd.DataFrame({
    "total_credit":total_credit,
    "total_debit":total_debit
}).reset_index()
df=pd.merge(df,solde, on="client_id")


In [ ]:
df["solde_net"]=(df["total_credit"]-df["total_debit"])
df.columns

In [ ]:
stats = df.groupby("client_id").agg(
    nb_transaction=("transaction_id","count"),
    montant_moyen=("montant","mean"),
    nb_produit=("produit","nunique")
).reset_index()
stats
stats=pd.DataFrame(stats)
df=pd.merge(df,stats,on="client_id").reset_index()

In [ ]:
df["taux_rejet"] = df.groupby('agence')['statut'].transform(
    lambda x: (x == 'Rejete').mean() * 100
)
df.head()

In [ ]:
df.to_csv("financecore_clean.csv",index=False)